# Text Preprocessing

This notebook prepares PubMed article titles and abstracts for biomedical
topic modeling.

The raw dataset contains recent PubMed abstracts on antibiotic resistance collected programmatically from PubMed.



In [1]:
from pathlib import Path
import re

import pandas as pd

## 1. Load Data

In [3]:
DATA_PATH = Path("../data/pubmed_antibiotic_resistance.csv")

df = pd.read_csv(DATA_PATH)

df.head()

,pmid,title,abstract,publication_year,journal,authors
0,41090395,Diagnostic accuracy of otitis media with and w...,BACKGROUND: Otitis media (OM) in children is a...,2026,Scandinavian journal of primary health care,"['Hedman M', 'Kosuta V', 'Lindmark M', 'Sandst..."
1,42498403,Plastic-mediated alterations in soil microbial...,The spatiotemporal co-accumulation of plastics...,2026,Journal of environmental sciences (China),"['Ye Y', 'Shen L', 'Lin D', 'Zhang T', 'Wang Y..."
2,42498393,Impact of antimicrobial peptide Hidefensin5 as...,Antibiotic resistance has arisen as a formidab...,2026,Journal of environmental sciences (China),"['Xia J', 'Lu Z', 'Ge C', 'Yao H']"
3,42498390,Release and bacterial transformation activity ...,Dissolved sulfides are widely distributed in a...,2026,Journal of environmental sciences (China),"['Yi L', 'Zhang W', 'Li H', 'Liu J', 'Zhang Z'..."
4,40892486,Molecular Characterization of beta-Lactamase-R...,The number of dairy farms in Bangladesh is ste...,2026,Foodborne pathogens and disease,"['Fahim FJ', 'Prome AA', 'Rana S', 'Uddin MS',..."


## 2. Checking the table

In [4]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Duplicate PMIDs: {df['pmid'].duplicated().sum()}")
print(f"Missing abstracts: {df['abstract'].isna().sum()}")
print(f"Missing titles: {df['title'].isna().sum()}")

Rows: 1,000
Columns: 6
Duplicate PMIDs: 0
Missing abstracts: 0
Missing titles: 0


In [5]:
df["publication_year"].value_counts().sort_index()

publication_year
2025    368
2026    632
Name: count, dtype: int64

In [6]:
df["abstract_length"] = df["abstract"].fillna("").str.len()

df["abstract_length"].describe().round(0)

count     1000.0
mean      1763.0
std        921.0
min        203.0
25%       1416.0
50%       1695.0
75%       1997.0
max      25195.0
Name: abstract_length, dtype: float64

In [7]:
df.nsmallest(10, "abstract_length")[
    ["pmid", "title", "abstract", "abstract_length"]
]

,pmid,title,abstract,abstract_length
901,41465417,Creation of New Antimicrobial Peptides 3: Rese...,The studies conducted and published in this is...,203
14,41170694,Antibiotic resistance and first comprehensive ...,"Identified 26 antibiotic resistance genes, inc...",278
232,41468546,Draft genome sequence of Pseudomonas aeruginos...,Pseudomonas aeruginosa is currently considered...,279
236,41416835,Draft genome sequences of two clinical isolate...,Candida tropicalis is an emerging human pathog...,282
586,41133786,The role of public hygiene behaviour in tackli...,Continuing on from their previously published ...,285
233,41432162,Draft genome sequence of a Klebsiella pneumoni...,"Klebsiella pneumoniae phage vB_KpBD_211, isola...",286
928,41251547,Draft genome sequence of Gleimia europaea DSM ...,"Here, we report the draft genome sequence of G...",291
930,41222165,Complete genome sequence of Pediococcus pentos...,"Pediococcus pentosaceus TOKAI 10 m, isolated f...",292
935,41165114,Complete genome sequence of Enterococcus faeci...,We report the complete genome of Enterococcus ...,305
933,41186221,Genome sequences of two highly colistin-resist...,The genomes of two highly colistin-resistant s...,315


Very short abstracts are likely to contain incomplete content and may reduce topic-model quality, so they are excluded from the final corpus.

In [17]:
sample_titles = df.sample(20, random_state=42)[["pmid", "title"]]
sample_titles

,pmid,title
521,41380256,Reductive soil disinfestation mitigates antibi...
737,41454319,Recombinant Listeria expressing the HPV multiv...
740,41923835,Patterns of multidrug resistance in Salmonella...
660,39809709,Dexamethasone-Antibiotic Interactions in Canin...
411,41164344,Native mass spectrometry of membrane proteins ...
678,29489203,Amoxicillin.
626,40961576,Metagenome assembled genomes revealed the infl...
513,41391320,Bacillus sp. S361 isolated from bioaerosols in...
859,41402626,Metabolomics-driven prediction of antibiotic r...
136,41839669,Bioactivity of green-synthesized zinc oxide na...


Manual inspection of a random sample suggests that the query retrieves a broader antibiotic-related corpus, not only papers directly focused on resistance. This is acceptable for exploratory topic modeling, but it should be noted in the project limitations.

## 3. Remove problematic entries

In [8]:
initial_rows = len(df)

df = (
    df.dropna(subset=["abstract"])
      .drop_duplicates(subset="pmid")
      .copy()
)

df["abstract"] = df["abstract"].astype(str).str.strip()

df = df[df["abstract"].str.len() >= 200].copy()

print(f"Rows before cleaning: {initial_rows:,}")
print(f"Rows after cleaning: {len(df):,}")
print(f"Removed rows: {initial_rows - len(df):,}")

Rows before cleaning: 1,000
Rows after cleaning: 1,000
Removed rows: 0


## 4. Combine the title and abstract

In [9]:
df["text"] = (
    df["title"].fillna("").astype(str)
    + ". "
    + df["abstract"].fillna("").astype(str)
)

df[["title", "abstract", "text"]].head(2)

,title,abstract,text
0,Diagnostic accuracy of otitis media with and w...,BACKGROUND: Otitis media (OM) in children is a...,Diagnostic accuracy of otitis media with and w...
1,Plastic-mediated alterations in soil microbial...,The spatiotemporal co-accumulation of plastics...,Plastic-mediated alterations in soil microbial...


## 5. Clean up the text

In [10]:
def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9\s\-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["clean_text"] = df["text"].apply(clean_text)
df[["text", "clean_text"]].head(2)

,text,clean_text
0,Diagnostic accuracy of otitis media with and w...,diagnostic accuracy of otitis media with and w...
1,Plastic-mediated alterations in soil microbial...,plastic-mediated alterations in soil microbial...


## 6. Check words and documents

In [11]:
df["n_words"] = df["clean_text"].str.split().str.len()

df["n_words"].describe().round(1)

count    1000.0
mean      254.7
std       130.2
min        40.0
25%       205.0
50%       249.0
75%       288.0
max      3548.0
Name: n_words, dtype: float64

In [12]:
df.nsmallest(10, "n_words")[
    ["pmid", "title", "n_words", "clean_text"]
]

,pmid,title,n_words,clean_text
901,41465417,Creation of New Antimicrobial Peptides 3: Rese...,40,creation of new antimicrobial peptides 3 resea...
586,41133786,The role of public hygiene behaviour in tackli...,49,the role of public hygiene behaviour in tackli...
14,41170694,Antibiotic resistance and first comprehensive ...,52,antibiotic resistance and first comprehensive ...
236,41416835,Draft genome sequences of two clinical isolate...,53,draft genome sequences of two clinical isolate...
928,41251547,Draft genome sequence of Gleimia europaea DSM ...,56,draft genome sequence of gleimia europaea dsm ...
667,32491763,Pseudomonas aeruginosa Infections.,57,pseudomonas aeruginosa infections pseudomonas ...
933,41186221,Genome sequences of two highly colistin-resist...,58,genome sequences of two highly colistin-resist...
232,41468546,Draft genome sequence of Pseudomonas aeruginos...,59,draft genome sequence of pseudomonas aeruginos...
233,41432162,Draft genome sequence of a Klebsiella pneumoni...,61,draft genome sequence of a klebsiella pneumoni...
930,41222165,Complete genome sequence of Pediococcus pentos...,62,complete genome sequence of pediococcus pentos...


In [13]:
assert df["clean_text"].notna().all()
assert (df["clean_text"].str.len() > 0).all()
assert df["pmid"].is_unique

## 7. Save a clean file

In [15]:
output_columns = [
    "pmid",
    "title",
    "abstract",
    "publication_year",
    "journal",
    "clean_text"
]

df_clean = df[output_columns].copy()

In [16]:
OUTPUT_PATH = Path("../data/pubmed_antibiotic_resistance_clean.csv")

df_clean.to_csv(OUTPUT_PATH, index=False)

print(f"Saved {len(df_clean):,} documents to {OUTPUT_PATH}")

Saved 1,000 documents to ..\data\pubmed_antibiotic_resistance_clean.csv


## Summary

The final cleaned corpus contains PubMed titles and abstracts prepared for
topic modeling. Texts were normalized, short abstracts were removed, and the
final dataset was saved for downstream NMF analysis.